In [ ]:
# from typing import Tuple

# import numpy as np
# import json


# import time

import numpy as np
sys.path.append('D:/Analysis_2P/suite2p')

# import  rigid
# import tifffile
# import matplotlib.pyplot as plt
# import numpy as np
# from caiman.mmapping import load_memmap
from pathlib import Path
# from scipy.ndimage import gaussian_filter
# from tifffile import TiffFile
from compute_zcorr import compute_zpos,compute_zcorr_for_frame

In [ ]:
# 1. load mmap
# 2. repalce z files with shifted z files
# 3. add the function into pipelne pipeline_mcorr_cnmf line 661
# 4. plot ther corrected z series

In [ ]:

# def compute_zpos(zstack_in, params_json, Treg_mmap_path):
#     """Compute z position of frames given z-stack Zreg and Treg array.

#     Parameters
#     ----------
#     Zreg_path : str
#         Path to the z-stack TIFF file.
#     params_json : str
#         Path to the JSON file containing parameters.
#     Treg_mmap_path : str
#         Memory-mapped file path for the registered T-series.

#     Returns
#     -------
#     zcorr : ndarray
#         Array of shape (Zreg.shape[0], nFrames) containing the z-correlation values.
#     """

#     Zreg_path = zstack_in+"/zstack4_shifted.tif"
#     # Load parameters from the JSON file
#     with open(params_json, 'r') as file:
#         params = json.load(file)

#     # Get the zcorr parameters
#     ops = params['zcorr']

#     # Load the z-stack
#     # zstack_tiff_file = tifffile.TiffFile(Zreg_path)
#     # Zreg = zstack_tiff_file.asarray()
#     # zstack_tiff_file.close()

#     zstack_tiff_file = TiffFile(Zreg_path)
#     Zreg = []
#     for page in zstack_tiff_file.pages:
#         Zreg.append(page.asarray())
#     Zreg = np.stack(Zreg, axis=0)
#     zstack_tiff_file.close()



#     print(Zreg.shape, Zreg.dtype, Zreg.max(), Zreg.min())
#     # Flip the Zreg
#     Zreg = np.flip(Zreg, axis=1)

#     # Load Treg_mmap_path
#     mcorr_movie_16bit, dims, T = load_memmap(Treg_mmap_path)
#     # Reshape the array to the desired dimensions
#     Treg = np.reshape(mcorr_movie_16bit.T, [T] + list(dims), order='F')
#     print(Treg.shape, Treg.dtype, Treg.max(), Treg.min())

#     Ly, Lx = Treg.shape[1], Treg.shape[2]


#     ops["nonrigid"] = False
#     nplanes, zLy, zLx = Zreg.shape
#     if Zreg.shape[1] > Ly or Zreg.shape[2] != Lx:
#         Zreg = Zreg[:, :Ly, :Lx]

#     nFrames, _, _ = Treg.shape
#     zcorr = np.zeros((Zreg.shape[0], nFrames), np.float32)
#     t0 = time.time()

#     refAndMasks = []
#     for Z in Zreg:
#         if ops["1Preg"]:
#             Z = Z.astype(np.float32)
#             Z = Z[np.newaxis, :, :]
#             if ops["pre_smooth"]:
#                 Z = utils.spatial_smooth(Z, int(ops["pre_smooth"]))
#             Z = utils.spatial_high_pass(Z, int(ops["spatial_hp_reg"]))
#             Z = Z.squeeze()

#         maskMul, maskOffset = rigid.compute_masks(
#             refImg=Z,
#             maskSlope=3 * ops["smooth_sigma"],
#         )
#         cfRefImag = rigid.phasecorr_reference(refImg=Z, smooth_sigma=ops["smooth_sigma"])
#         cfRefImag = cfRefImag[np.newaxis, :, :]
#         refAndMasks.append((maskMul, maskOffset, cfRefImag))

#     for nfr in range(nFrames):
#         data = Treg[nfr:nfr + 1]  # Get the current frame
#         inds = np.array([nfr])     # Index of the current frame

#         for z, ref in enumerate(refAndMasks):
#             if ops["1Preg"]:
#                 data = data.astype(np.float32)
#                 if ops["pre_smooth"]:
#                     data = utils.spatial_smooth(data, int(ops["pre_smooth"]))
#                 data = utils.spatial_high_pass(data, int(ops["spatial_hp_reg"]))

#             maskMul, maskOffset, cfRefImg = ref
#             cfRefImg = cfRefImg.squeeze()

#             _, _, zcorr[z, inds] = rigid.phasecorr(
#                 data=rigid.apply_masks(data=data, maskMul=maskMul, maskOffset=maskOffset),
#                 cfRefImg=cfRefImg,
#                 maxregshift=ops["maxregshift"],
#                 smooth_sigma_time=ops["smooth_sigma_time"],
#             )

#         if nfr % 10 == 0:
#             print(f"{z + 1} planes, {nfr + 1}/{nFrames} frames, {time.time() - t0:.2f} sec.")

#     print(f"{z + 1} planes, {nFrames}/{nFrames} frames, {time.time() - t0:.2f} sec.")


#     # Apply Gaussian smoothing along the Z dimension
#     zcorr = gaussian_filter(zcorr, sigma=[1, 0])  # Adjust sigma[0] for the desired smoothing level

#     return zcorr

In [ ]:
# ops = {         
    
#     'smooth_sigma': 1.15,         # Standard deviation for Gaussian smoothing of Z-stack
#     '1Preg': False,              # False for two-photon data
#     'pre_smooth': 0,             # Pre-smoothing is not typically used for 2P data
#     'spatial_hp_reg': 42,         # High-pass filtering is not typically used for 2P data
#     'spatial_taper': 40,         # Width of the taper applied to the edges of the frames
#     'maxregshift': 0.1,           # Maximum allowed shift in pixels during registration
#     'smooth_sigma_time': 0,      # Standard deviation for temporal smoothing of registration shifts
#     'nonrigid': False,           # False for rigid registration
# }

# Zreg_path=  "D:/Analysis_2P/Data/Scnn1aAi14_A2M0/01042024/ZSeries-01042024-1531-001/zstack4_shifted.tif"
Zreg_path = "D:/Analysis_2P/Data/C57_O1M2/10022023/ZSeries-10022023-1300-002/"

proc_folder = Path('D:/Analysis_2P/Data/Analysis/C57_O1M2/10022023/run2/mesmerize')
mcorr_folder = '19fa5a4a-0d1d-4063-b67d-eee4a56e8a14'
mcorr_mmap = f"{mcorr_folder}-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_1600.mmap"
Treg_mmap_path = proc_folder / mcorr_folder / mcorr_mmap
params_json = "D:/Analysis_2P/Analysis_2P/Mesmerize/parameters/params_zshift_default.json"


In [ ]:
zcorr = compute_zpos(Zreg_path,params_json,Treg_mmap_path) 

In [ ]:
zcorr.shape

In [ ]:
zcorr

In [ ]:

# Apply Gaussian smoothing along the Z dimension
# smoothed_zcorr = gaussian_filter(zcorr, sigma=[0,1])  # Adjust sigma[0] for the desired smoothing level


In [ ]:
z_positions = np.argmax(zcorr, axis=0)
print(z_positions)

In [ ]:
import matplotlib.pyplot as plt

def plot_zcorr(zcorr):
    plt.plot(zcorr)
    plt.xlabel('Frames')
    plt.ylabel('Z Position')
    plt.title('Line Chart of zcorr')
    plt.grid(True)
    plt.show()

# Example usage
plot_zcorr(z_positions)

In [ ]:
zcorr

In [ ]:
# compare suite2p data
import numpy as np
from scipy.io import loadmat

def load_mat_to_ndarray(file_path, variable_name):
    """
    Load a specified variable from a .mat file and convert it to a NumPy ndarray.
    
    Parameters:
    - file_path: str, the path to the .mat file.
    - variable_name: str, the name of the variable in the .mat file to convert to an ndarray.
    
    Returns:
    - ndarray: The specified variable as a NumPy ndarray.
    """
    # Load the .mat file
    data = loadmat(file_path)
    
    # Extract the specified variable
    if variable_name in data:
        ndarray = data[variable_name]
        # Check if the loaded data is structured or requires further processing
        if isinstance(ndarray, np.ndarray):
            return ndarray
        else:
            raise ValueError(f"{variable_name} is not stored as an ndarray.")
    else:
        raise KeyError(f"{variable_name} not found in the .mat file.")




In [ ]:
# Usage example

file_path = "C:/Users/Kyle/Downloads/zcorr (1).mat"
ndarray = load_mat_to_ndarray(file_path,"zcorr")
print(ndarray)

In [ ]:
ndarray.shape

In [ ]:
z_positions_suite2p = np.argmax(ndarray, axis=0)
print(z_positions_suite2p)

In [ ]:
# Example usage
plot_zcorr(z_positions_suite2p)

In [ ]:
# plot two figures together
plt.plot(z_positions, label='caiman')
plt.plot(z_positions_suite2p, label='suite2p')
plt.xlabel('Frames') 
plt.ylabel('Z Position')
plt.title('zcorr between caiman and suite2p')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# plot two figures together
plt.plot(z_positions, label='caiman')
plt.plot(z_positions_suite2p, label='suite2p')
plt.xlabel('Frames') 
plt.ylabel('Z Position')
plt.title('zcorr between caiman and suite2p')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# load mat file
import numpy as np
from scipy.io import loadmat
# LOAD 

In [ ]:

# 找出不同值的索引
different_indices = [i for i, (a, b) in enumerate(zip(z_positions, z_positions_suite2p)) if a != b]
print("z_positions_caiman: ", z_positions)
print("z_positions_suite2p: ", z_positions_suite2p)
print("Index of difference frames: ", different_indices)
print(len(different_indices))

In [ ]:
import scipy.io

# Assuming z_positions and z_positions_suite2p are defined
different_indices = [i for i, (a, b) in enumerate(zip(z_positions, z_positions_suite2p)) if a != b]

# Create a dictionary to store the data
mat_data = {
    'z_positions_caiman': z_positions,
    'z_positions_suite2p': z_positions_suite2p,
    'index_of_difference_frames': different_indices
}


# Save the dictionary to a .mat file
scipy.io.savemat('difference_indices_10022023_1300_002.mat', mat_data)

print("Results have been saved to difference_indices.mat")





In [ ]:
# load it back
import scipy.io
import numpy as np
mat_data = scipy.io.loadmat('difference_indices_10022023_1300_002.mat')
z_positions_caiman = mat_data['z_positions_caiman']
z_positions_suite2p = mat_data['z_positions_suite2p']

# compute the error rate between two z positions
error_rate = np.mean(z_positions_caiman != z_positions_suite2p)
print("error_rate",error_rate)

# compute the rate that the difference is within 1 frame
within_1_frame_rate = np.mean(abs(z_positions_caiman - z_positions_suite2p) > 1)
print("diff_exceed_1_frame_rate",within_1_frame_rate)


# compute the rate that the difference is within 2 frames
within_2_frames_rate = np.mean(abs(z_positions_caiman - z_positions_suite2p) > 2)
print("diff_exceed__2_frame_rate",within_2_frames_rate)



# compute the rate that the difference is within 3 frames
within_3_frames_rate = np.mean(abs(z_positions_caiman - z_positions_suite2p) > 3)
print("diff_exceed__3_frame_rate",within_3_frames_rate)
